# 03 · ML counterfactual: what would 2025 have looked like without the toll?

**Approach:** train XGBoost on 2023–2024 daily Uber trips for each zone group (inside, entering, exiting), using calendar features, holidays, a time trend, and same-day trips in the uptown Manhattan control. Then predict 2025 onward *as if there were no toll*. **Toll effect = actual − predicted.**

**Credibility check (placebo):** train through June 2024 only, predict July–December 2024 (no toll yet). The "effect" there should be about 0.

In [ ]:
import os, sys
os.chdir(r"C:\Users\lalit\Documents\nyc-congestion-toll-impact")
sys.path.append("src")

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from pandas.tseries.holiday import USFederalHolidayCalendar
from zones import CBD_ZONES, BOUNDARY_ZONES

pd.set_option("display.width", 200)

## 1. Daily trips per zone group + uptown control

In [ ]:
lookup = pd.read_csv("data/taxi_zone_lookup.csv")
uptown = lookup[(lookup["Borough"] == "Manhattan")
                & (~lookup["LocationID"].isin(CBD_ZONES + BOUNDARY_ZONES))
                & (~lookup["LocationID"].isin([103, 104, 105]))]
uptown_ids = ",".join(map(str, uptown["LocationID"]))

raw = duckdb.sql(f"""
    SELECT trip_date,
           CASE WHEN trip_group = 'outside' THEN 'control' ELSE trip_group END AS grp,
           SUM(trips) AS trips
    FROM 'data/summary/*.parquet'
    WHERE company = 'Uber'
      AND (trip_group IN ('inside', 'entering', 'exiting')
           OR (trip_group = 'outside' AND pu_zone IN ({uptown_ids})))
    GROUP BY ALL
""").df()
raw["trip_date"] = pd.to_datetime(raw["trip_date"])
print(raw["trip_date"].min().date(), "to", raw["trip_date"].max().date(), "|", len(raw), "rows")

## 2. Features

In [ ]:
wide = raw.pivot(index="trip_date", columns="grp", values="trips").sort_index()
hol = USFederalHolidayCalendar().holidays("2023-01-01", "2026-12-31")

GROUPS = ["inside", "entering", "exiting"]
df = wide[GROUPS].stack().rename("trips").reset_index()
df.columns = ["trip_date", "grp", "trips"]
df = df.merge(wide["control"].rename("control_trips"), left_on="trip_date", right_index=True)

df["dow"] = df["trip_date"].dt.dayofweek
df["month"] = df["trip_date"].dt.month
df["doy"] = df["trip_date"].dt.dayofyear
df["week"] = df["trip_date"].dt.isocalendar().week.astype(int)
df["is_holiday"] = df["trip_date"].isin(hol).astype(int)
df["near_holiday"] = df["trip_date"].apply(lambda d: int(any(abs((d - h).days) <= 2 for h in hol)))
df["holiday_season"] = ((df["month"] == 12) & (df["trip_date"].dt.day >= 20) |
                        (df["month"] == 1) & (df["trip_date"].dt.day <= 3)).astype(int)
df["t"] = (df["trip_date"] - pd.Timestamp("2023-01-01")).dt.days
df["log_control"] = np.log(df["control_trips"])
df["grp_code"] = df["grp"].map({g: i for i, g in enumerate(GROUPS)})
df["y"] = np.log(df["trips"])

FEATURES = ["grp_code", "dow", "month", "doy", "week", "is_holiday", "near_holiday",
            "holiday_season", "t", "log_control"]

def fit(train):
    m = XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=4,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
    m.fit(train[FEATURES], train["y"])
    return m

df.shape

## 3. Placebo test: train through Jun 2024, predict Jul–Dec 2024 (no toll)

In [ ]:
train = df[df["trip_date"] < "2024-07-01"]
test = df[(df["trip_date"] >= "2024-07-01") & (df["trip_date"] < "2025-01-01")].copy()
m = fit(train)
test["pred"] = np.exp(m.predict(test[FEATURES]))

placebo = test.groupby("grp").apply(lambda g: pd.Series({
    "actual": g["trips"].sum(),
    "predicted": g["pred"].sum(),
    "placebo_effect_pct": (g["trips"].sum() / g["pred"].sum() - 1) * 100,
    "MAPE_daily_pct": (abs(g["trips"] - g["pred"]) / g["trips"]).mean() * 100,
}), include_groups=False).round(1)
placebo.to_csv("outputs/ml_placebo_uber.csv")
placebo

## 4. The counterfactual: train on 2023–2024, predict 2025+

In [ ]:
train = df[df["trip_date"] < "2025-01-01"]
model = fit(train)
df["pred"] = np.exp(model.predict(df[FEATURES]))

post = df[df["trip_date"] >= "2025-01-05"].copy()
post["year"] = post["trip_date"].dt.year
effect = post.groupby(["year", "grp"]).apply(lambda g: pd.Series({
    "actual_M": g["trips"].sum() / 1e6,
    "no_toll_M": g["pred"].sum() / 1e6,
    "effect_pct": (g["trips"].sum() / g["pred"].sum() - 1) * 100,
    "trips_lost_K_per_month": (g["pred"].sum() - g["trips"].sum())
                              / g["trip_date"].dt.to_period("M").nunique() / 1e3,
}), include_groups=False).round(1)
effect.to_csv("outputs/ml_counterfactual_uber.csv")
effect

## 5. Chart: actual vs no-toll prediction (all zone groups)

In [ ]:
wk = df.groupby(pd.Grouper(key="trip_date", freq="W"))[["trips", "pred"]].sum()
wk = wk.iloc[1:-1] / 1e6    # drop partial first/last weeks

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(wk.index, wk["trips"], color="#222222", lw=1.6, label="Actual")
ax.plot(wk.index, wk["pred"], color="#d62728", lw=1.6, ls="--", label="Model (no toll after 2025)")
ax.axvline(pd.Timestamp("2025-01-05"), color="black", ls=":", lw=1)
ax.axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31"), color="grey", alpha=0.06)
ax.text(pd.Timestamp("2023-02-01"), ax.get_ylim()[0] * 1.02, "Training period", fontsize=10)
ax.set_title("Uber weekly trips touching the congestion zone: actual vs ML counterfactual")
ax.set_ylabel("Trips (millions per week)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/ml_counterfactual_uber.png", dpi=150)
plt.show()